# Week 3 (together), Modelling: logistic regression, built by the room

**Today you do the modelling, and modelling is a series of choices.** Which two piles. What
counts as a feature. What score you'd believe. Which of several models you keep, and why. If
the code were already written you'd read those choices instead of making them, so the cells
below are empty.

**Blank does not mean unguided.** Each station explains the idea, names the tools to use, and
tells you how to check whether the result is right. It doesn't give you the code. That part is
yours and the AI's.

**How to read a cell.** Any code cell that holds nothing but comments is yours to fill — one
task per cell, so nobody is ever staring at a blank page wondering where to start. Lines marked
`PROMPT IDEA:` are a suggested first message to the AI; send it, then **read what comes back
before you run it** — that reading is the Reader's job and it is most of the learning. Lines in
CAPITALS are answers to write down in the comment itself, so your group leaves with a record.

**What your group leaves with:** a number you can defend, two word lists you can explain, a
model you broke on purpose, and one caveat you'd put in writing.

### Saying what you found

Each station ends with numbers on screen and a decision about what they mean. Before you say
anything, check you have four things:

1. **The claim**, in one sentence, with no numbers in it.
2. **The number** that would change your mind if it moved.
3. **What it covers**: which two piles, how many comments, from when.
4. **The best objection**, said by you first.

So each station tells you what to look at, then gives you a sentence to fill in. If you can't
fill all four parts, say that. "We got 0.74 and we don't know yet what it means" is a fine
thing to report.

---

### How to work

Threes, one screen, and rotate these three jobs at every station:

| Job | Does |
|---|---|
| **Driver** | Types, and prompts the AI. Never types a line nobody has read aloud. |
| **Reader** | Says what the cell will do *before* it runs, then whether it did. |
| **Skeptic** | Asks the awkward question. Is that better than guessing? Would that word survive on someone else's data? |

### The stations

| # | Station | In the 32-minute block |
|---|---|---|
| 0 | Warm-up: the whole pipeline on six toy sentences (written for you) | 3 min |
| 1 | The question, and your prediction | 3 min |
| 2 | Features: turn text into numbers | 4 min |
| 3 | Fit the model | 4 min |
| 4 | Judge it honestly | 5 min |
| 5 | Read its mind | 5 min |
| 6 | Break it | 4 min |
| 7 | **The workbench:** run many configurations, one change at a time | 6 min |
| 8 | **Interesting weights:** interrogate one word — is it real? | pick one |
| 9 | **Change the corpus:** the same model on two other communities | pick one |
| — | **Open bench:** three empty cells and the question your group actually wants to ask | after class |
| 10 | Report back | 4 min |

**There is deliberately more here than fits in the block.** Stations 0–7 and 10 are the class;
8 and 9 are a choose-one in the room and the natural place to keep going afterwards, since the
sketch homework is exactly this notebook pointed at data you care about. Stations 7, 8 and 9
come with the plumbing written for you — by then you have built the pipeline once by hand, and
the six minutes should go on choices, not typing.

Behind schedule? Stations 4, 5 and 7 are the ones that must happen. The worked version of this
pipeline is `week03_classification.ipynb` — peek if you're stuck, after you've asked the AI and
read what it gave you.

> **Don't lose your work.** Opened from GitHub, this notebook is read-only: **File → Save a copy in Drive** before editing, and save durable outputs to your Drive project folder; Colab's own disk is wiped when the runtime ends. Course notebooks get updates during the term; to pick them up, open the notebook fresh from GitHub (or `git pull` if you cloned the repo). Updates never touch your saved copy.

In [ ]:
# If an import fails: re-run this cell; if it persists see ../kits/common-errors-cheatsheet.md
# (standalone copy: https://github.com/lucianli123/culture-as-data-2026/blob/main/kits/common-errors-cheatsheet.md)
# --- Make your work survive a Colab reset -------------------------------------
# Colab wipes the runtime when it disconnects or idles out. Mount your Google Drive
# and keep everything in ONE project folder, so your corpus, models, and charts are
# still there next week. (Outside Colab - e.g. the offline test harness - this falls
# back to a local folder so the notebook still runs.)
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/culture-as-data"
except Exception:
    PROJECT_DIR = os.path.abspath("./culture-as-data-project")
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder:", PROJECT_DIR)

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
print("imports ok")

---

## Station 0 · The whole pipeline, on six sentences (3 min, written for you)

Before you build anything on real data, watch the entire method happen on a corpus small enough
to hold in your head: six sentences, three about the sea and three about the kitchen.

Run the next cell, then read it line by line. **Everything you do for the rest of the session is
this cell, at scale, with your own choices substituted.** The four moving parts, in order:

1. **Features.** `CountVectorizer` builds a vocabulary from the corpus and turns each document
   into a row of counts. Rows are documents, columns are words.
2. **Weights.** `LogisticRegression` learns one number per column — positive pushes toward one
   label, negative toward the other, near-zero means the word carried no information.
3. **The fit.** Fitting *is* the search for those weights: the set that leans the right way for
   as many training documents as possible.
4. **Reading it.** `model.coef_` is the weights, `vectorizer.get_feature_names_out()` is the
   words. Line them up, sort, and you are reading the model's mind.

Predict before you run: which words will get the biggest weights?

In [ ]:
# --- The toy corpus: six documents, two labels --------------------------------
toy = pd.DataFrame({
    "text": ["the tide came in over the cold sand",
             "salt spray and a grey sea all morning",
             "we walked the shore until the tide turned",
             "he chopped onions and let the butter brown",
             "the bread proved overnight on the warm counter",
             "she salted the water and dropped the pasta in"],
    "label": ["sea", "sea", "sea", "kitchen", "kitchen", "kitchen"],
})

# 1. FEATURES: text -> a matrix of word counts.
toy_vec = CountVectorizer()
Xt = toy_vec.fit_transform(toy["text"])       # rows = documents, columns = words
yt = (toy["label"] == "sea").astype(int)      # the target: 1 = sea, 0 = kitchen
print("matrix shape (documents, vocabulary):", Xt.shape)
print("first ten of the vocabulary:", list(toy_vec.get_feature_names_out()[:10]))

# 2 + 3. WEIGHTS + FIT: one number per word, learned from the labeled examples.
toy_clf = LogisticRegression(max_iter=1000).fit(Xt, yt)

# 4. READ IT: line the words up against their weights and sort.
toy_words = np.array(toy_vec.get_feature_names_out())
toy_w = toy_clf.coef_.ravel()
order = toy_w.argsort()
print("\nmost KITCHEN:", ", ".join(toy_words[order[:4]]))
print("most SEA:     ", ", ".join(toy_words[order[-4:][::-1]]))

# Four weights worth looking at by name (the markdown below says why):
for word in ["in", "the", "salt", "salted"]:
    print(f"  weight of {word!r:9}", round(toy_w[list(toy_words).index(word)], 3))
print("accuracy on its OWN training rows:", toy_clf.score(Xt, yt), "<- meaningless")

# And the model as a reader: hand it a sentence it has never seen.
for s in ["the salt water boiled", "the grey morning sand"]:
    p = toy_clf.predict_proba(toy_vec.transform([s]))[0, 1]
    print(f"{s!r:28} -> {'sea' if p > 0.5 else 'kitchen'} (p_sea={p:.2f})")

**Read the output before you move on.** Three things worth noticing, because all three come
back at full scale:

- **Uninformative words earn small weights on their own.** Look up *in*: it appears once on
  each side, and its weight is essentially zero. The model needs no stop list to ignore it.
  But look up *the* as well — it lands a small non-zero weight, because in six documents it
  happens to fall slightly more on one side. With this little evidence, noise gets a weight
  too, and that never entirely stops being true.
- **The model has no idea that *salt* and *salted* are the same word.** They're separate
  columns here, pushing in opposite directions, one from the sea sentences and one from the
  kitchen. That's the exact run/running argument you had by hand in Week 2, now visible as two
  numbers.
- **Its accuracy on its own training rows is 100 percent, and means nothing.** Six documents,
  thirty-six columns: it can memorise. Nothing has been tested on data it didn't see. Station 4
  is where that gets fixed, and it's the single most common way a real result goes wrong.

One vocabulary note for the stations ahead: **features** are the columns (here, words),
**weights** (or coefficients) are the learned numbers, **fitting** or **training** is the
search for them, and **held-out** data is rows the model never saw during that search.

---

## Your corpus (written for you): two labelled piles

A classifier needs examples someone has already sorted. `load_pair()` hands your group a
DataFrame with two columns, `text` and `label`, and nothing else. Collecting a corpus is Week
4's whole subject; today it's plumbing.

**Your one setup choice: which two piles.** Set `PAIR` in the next cell.

| Pair | What it teaches |
|---|---|
| `("sandiego", "bayarea")` | The default: two California regions. Clear enough that the mechanics land — expect roughly 0.70 against a 0.50 baseline — and the top words are readable at a glance. Watch what *kind* of word wins. |
| Any two subreddits you know | `("coffee", "espresso")`, `("AskMen", "AskWomen")`, `("nba", "soccer")`, `("Seattle", "Portland")`. The archive covers essentially all of Reddit; comments are pulled live and analyzed only, never redistributed. |
| `("sandiego", "SanDiegan")` | **Hard mode.** Two communities about the *same* city — same beaches, same rents, largely the same words. The same code lands near 0.60, and the weights get genuinely hard to read. Worth trying once you have the easy case working. |
| `("shelley", "stoker")` | Sentences from *Frankenstein* and *Dracula*. No network needed, and the automatic fallback if the archive is slow. |

**Choose deliberately, and expect the default to teach you something blunt.** Two unrelated
topics gets you 95 percent accuracy and a boring reading: the words are just the two subjects.
The near-identical pair gets you 60 percent, and the words are hard-won. The default sits in
between on purpose — high enough that nothing is ambiguous, low enough that the words are worth
arguing about. And a model that barely beats a coin flip is still a *result*: it says these two
communities write alike, as long as you can show the baseline it beat.

In [ ]:
# --- Provided loader. Read it, don't rewrite it. ------------------------------
PAIR = ("sandiego", "bayarea")   # <-- your group's two piles (see the table above)
N_PER_SIDE = 400                   # rows per pile; 400 trains in a couple of seconds
MIN_PER_SIDE = 60                  # below this, a model isn't worth fitting
ARCHIVE = "https://arctic-shift.photon-reddit.com/api/comments/search"

def fetch_comments(sub, n=N_PER_SIDE, max_pages=12, tries=4):
    """Comments from one subreddit, analyze-only. Returns (texts, note).
    Keeps what it has already collected across retries, so a hiccup on page 7
    doesn't throw away pages 1-6."""
    import random, time, requests
    got, before, pages, attempt, dead = [], None, 0, 0, 0
    time.sleep(random.uniform(0, 2.0))          # stagger: the whole room starts at once
    while len(got) < n and pages < max_pages and attempt < tries:
        params = {"subreddit": sub, "limit": 100, "fields": "body,created_utc"}
        if before: params["before"] = before
        try:
            resp = requests.get(ARCHIVE, params=params, timeout=30)
        except Exception as e:
            attempt += 1; dead += 1
            if dead >= 2:                       # no network, or the archive is down
                return got, "unreachable"
            print(f"  r/{sub}: {type(e).__name__}, retry {attempt}/{tries}")
            time.sleep(min(8, 2 ** attempt) + random.uniform(0, 1))
            continue
        dead = 0
        if resp.status_code == 200:
            rows = resp.json().get("data") or []
            pages += 1
            if not rows:                        # no such subreddit, or we reached the end
                return got, ("empty" if not got else "ok")
            before = int(min(r["created_utc"] for r in rows))
            got += [r["body"] for r in rows if isinstance(r.get("body"), str)
                    and 80 < len(r["body"]) < 800
                    and "[removed]" not in r["body"] and "[deleted]" not in r["body"]
                    and "moderator" not in r["body"].lower()
                    and "has been removed" not in r["body"].lower()]
            continue
        if resp.status_code == 400:             # our request is malformed; retrying won't help
            return got, f"rejected: {resp.text[:80]}"
        attempt += 1                            # 422 "slow down", 429, 5xx: wait and retry
        wait = float(resp.headers.get("Retry-After") or min(8, 2 ** attempt))
        print(f"  r/{sub}: HTTP {resp.status_code}, waiting {wait:.0f}s (retry {attempt}/{tries})")
        time.sleep(wait + random.uniform(0, 1))
    return got, ("ok" if got else "failed")

def load_novelists(n=N_PER_SIDE):
    """The offline pair: Frankenstein against Dracula. Repo snapshots first,
    Project Gutenberg if you opened this notebook on its own."""
    def sents(text):
        return [s.strip().replace("\n", " ") for s in re.split(r"(?<=[.!?])\s+", text)
                if 40 < len(s) < 180][:n]
    def read_one(fname, url):
        for base in ("data/texts", "notebooks/data/texts", "../notebooks/data/texts"):
            path = os.path.join(base, fname)
            if os.path.exists(path):
                return open(path, encoding="utf-8", errors="ignore").read()
        import requests
        raw = requests.get(url, timeout=30).text.replace("\r\n", "\n")
        body = re.split(r"\*\*\* ?START OF (?:THE|THIS) PROJECT GUTENBERG.*?\*\*\*", raw, flags=re.S)[-1]
        return re.split(r"\*\*\* ?END OF (?:THE|THIS) PROJECT GUTENBERG", body)[0]
    a = sents(read_one("frankenstein.txt", "https://www.gutenberg.org/cache/epub/84/pg84.txt"))
    b = sents(read_one("dracula.txt", "https://www.gutenberg.org/cache/epub/345/pg345.txt"))
    return pd.DataFrame({"text": a + b, "label": ["shelley"] * len(a) + ["stoker"] * len(b)})

def load_pair(pair=PAIR, n=N_PER_SIDE):
    """Returns (df, label_a, label_b). Falls back to the novelists LOUDLY, never
    silently, and saves your pull to Drive so a Colab reconnect costs you nothing."""
    a, b = pair
    if {a, b} == {"shelley", "stoker"}:
        return load_novelists(n), "shelley", "stoker"
    cache = os.path.join(PROJECT_DIR, f"week03_{a}_vs_{b}.csv")
    if os.path.exists(cache):
        d = pd.read_csv(cache)
        print(f"loaded {len(d)} rows from your saved copy: {cache}")
        print("(delete that file if you want a fresh pull)")
        return d, a, b
    piles, notes = {}, {}
    for sub in (a, b):
        rows, note = fetch_comments(sub, n)
        piles[sub], notes[sub] = rows, note
        print(f"r/{sub}: {len(rows)} comments ({note})")
        if note == "unreachable":
            print("\nThe archive isn't answering - no network, or it's down. Falling back to")
            print("the novelists so you can keep working; the modelling is identical, and you")
            print("can re-run this cell with your own pair later.")
            return load_novelists(n), "shelley", "stoker"
        if note == "empty":
            print(f"   ^ nothing came back for r/{sub}. Check the spelling - that is")
            print(f"     almost always what an empty pile means.")
    if min(len(piles[a]), len(piles[b])) < MIN_PER_SIDE:
        thin = ", ".join("r/" + s for s in (a, b) if len(piles[s]) < MIN_PER_SIDE)
        print(f"\nToo few comments on {thin} (under {MIN_PER_SIDE}), so a model would be noise.")
        print("Falling back to the novelists. Fix PAIR" +
              (" - check the spelling - " if any(notes[s] == "empty" for s in (a, b)) else " ") +
              "and re-run to try again.")
        return load_novelists(n), "shelley", "stoker"
    d = pd.DataFrame({"text": piles[a] + piles[b],
                      "label": [a] * len(piles[a]) + [b] * len(piles[b])})
    big, small = max(len(piles[a]), len(piles[b])), min(len(piles[a]), len(piles[b]))
    if big > 2 * small:
        print(f"\nHeads up: uneven piles ({big} vs {small}). Your Station 4 baseline will be")
        print("high because of it, and class_weight='balanced' is worth a try at Station 7.")
    d.to_csv(cache, index=False)
    print(f"saved to {cache} - yours to analyze, not to redistribute")
    return d, a, b

df, LABEL_A, LABEL_B = load_pair()
print(df["label"].value_counts().to_string())
df.sample(4, random_state=1)

**Before you model, look.** Two numbers and one habit:

- How many rows per side? If one pile is much bigger, remember that number — it comes back at
  Station 4 as the score a lazy model gets for free.
- Read two comments aloud, in full. Everything you can only learn by looking, you learn now.
  Half the surprises at Station 5 are things a human would have spotted in thirty seconds of
  reading (a bot posting the same template, a pinned thread, one loud user).

### Asking the AI well

The AI writes today's code; you make today's decisions. Prompts that work are small and
concrete, and name the objects you already have:

> *"Using the DataFrame `df` with columns text and label, vectorize the text with
> CountVectorizer and show me the matrix shape."*

> *"Split that into train and test with 25 percent held out, stratified by y, random_state 0.
> Then fit a LogisticRegression and print the accuracy on the held-out part."*

> *"What does min_df=3 change about the matrix, in one sentence?"*

Prompts that don't: *"do week 3 for me."* You get code you can't defend at report-back, and the
Reader has nothing to narrate. Read every line back before running it — that's the job.

---

## Station 1 · The question, and your prediction (3 min)

One sentence, written down: what would it *mean* if a machine could tell these two piles apart,
and what would it mean if it couldn't? "Can a classifier do it" is not yet a question about
culture; "do these two communities write differently enough that a machine can hear it" is.

Then predict, out loud and in writing: what accuracy do you expect, and three words you think
will give each side away. A prediction made after seeing the result is not a prediction — and
being wrong here is the most useful thing that can happen to you today, because a surprise is
the only reliable sign you've learned something about the corpus rather than about sklearn.

In [ ]:
# OUR QUESTION: ...one sentence...
# OUR PREDICTION: accuracy about ..., because ...
# GIVEAWAY WORDS WE EXPECT: side A: ..., ..., ...   side B: ..., ..., ...

In [ ]:
# Look before you predict: print two full-length comments from each pile.
# PROMPT IDEA: "print two full rows of df['text'] for each value of df['label'], no truncation"

---

## Station 2 · Features: turn text into numbers (4 min)

A logistic regression cannot see text. Something has to turn each document into a row of
numbers, and that something is your first modelling choice — made, if you don't make it,
by the defaults.

`CountVectorizer` builds a vocabulary from your corpus and counts. Its defaults quietly decide
a great deal: it lowercases everything (so *Reddit* and *reddit* are one column), splits on a
pattern that drops punctuation and single characters, keeps every word that survives, and
treats *run* and *running* as unrelated columns. Every one of those was a decision Week 2 made
you argue about by hand.

**Build it.** Turn `df["text"]` into a matrix `X`, and `df["label"]` into a 0/1 target `y`.
Then print the shape and a slice of the vocabulary.

**Tools:** `CountVectorizer()`, `.fit_transform(...)`, `.get_feature_names_out()`, and
`(df["label"] == LABEL_A).astype(int)` for the target.

**Look at three things.**

- **Rows and columns.** About 800 rows, a few thousand columns.
- **Columns divided by rows.** Usually 3 to 8 columns per document. Remember that number, it
  comes back at Station 4.
- **The vocabulary sample.** Look for numbers, bits of URLs, single letters, the same word
  spelled two ways. Each one is a column the model can use.

**Then fill this in:**

> "Each comment became a row of ___ numbers, one per word. The vectorizer lowercased
> everything, dropped punctuation, and treated *run* and *running* as different words. We took
> those defaults rather than choosing them."

**Say this before anyone asks:** there are more columns than documents, so the model has room to
memorise. Station 4 checks whether it did.

In [ ]:
# 2a. Turn df["text"] into a matrix X.
# PROMPT IDEA: "vectorize df['text'] with CountVectorizer into X, and show me X.shape"

In [ ]:
# 2b. Build the target y: 1 for LABEL_A, 0 for LABEL_B.

In [ ]:
# 2c. Look at the vocabulary the vectorizer built: the first 20 names, and a few
#     from the middle of the alphabet. Anything in there that surprises you?


# HOW MANY COLUMNS: ...
# THE DEFAULT WE'D CHANGE, AND WHY: ...

---

## Station 3 · Fit the model (4 min)

Before you run it, the Reader says what fitting does: **the model is searching for one weight
per word, such that the weighted sum of a document's words leans the right way for as many
training documents as possible.** No understanding, no rules anyone wrote — a few thousand
numbers, tuned until the votes come out right.

Two things to get right here, because everything downstream depends on them:

- **Hold data out.** `train_test_split` with `test_size=0.25` keeps a quarter of the rows away
  from training so you have something honest to score on. Use `stratify=y` so both piles are
  represented in both halves, and `random_state=0` so your Station 7 comparisons are against
  the same split rather than a different roll of the dice.
- **Let it converge.** `LogisticRegression(max_iter=1000)`. The default iteration cap is low
  for text-sized vocabularies, and the warning you'd otherwise get means "the optimiser ran out
  of steps", not "your model is wrong".

**Look at two numbers and the gap between them.**

- **Held-out accuracy.** Anything from 0.55 to 0.99 is possible. On the default pair, expect
  about 0.75.
- **Accuracy on the training rows.** It will be close to 1.00. That is memory, not a result.
- **The gap between them.** A big gap means the model memorised. You will make that gap on
  purpose at Station 7 by turning `C` up.

**Then fill this in:**

> "On rows it never saw, the model told the two piles apart ___% of the time."

That sentence is half finished. The other half is what it beat, and that is Station 4.

**One warning:** if held-out accuracy is 1.00, stop and look for a giveaway in the text — a
subreddit name, a bot template, a scraping artifact. Finding that is worth more than the 1.00.

In [ ]:
# 3a. Split off a quarter the model will never see.
# PROMPT IDEA: "split X and y into train and test, 25% held out, stratified by y,
#               random_state=0"

In [ ]:
# 3b. Fit a LogisticRegression on the training half. Reader: say what fitting does
#     BEFORE this runs.

In [ ]:
# 3c. Score it on the held-out quarter, and print the number.


# HELD-OUT ACCURACY: ...
# ANYTHING SUSPICIOUS ABOUT IT (1.00 is a red flag, not a triumph): ...

In [ ]:
# 3d. Refit a second copy on ALL the rows. That is the one to read weights from at
#     Station 5, because it has seen the most evidence. Keep the two straight.

---

## Station 4 · Judge it honestly (5 min)

An accuracy number on its own means nothing. Three things give it meaning, and this station is
where most published mistakes would have been caught.

**1. A baseline.** What does a model that always guesses the bigger pile score? If your piles
are 400/364, that's 52 percent for free, without reading a word. Your model's number is only
interesting as a *distance above* that floor. `DummyClassifier(strategy="most_frequent")` fits
and scores exactly like a real model, which is the point — it's the null hypothesis you can run.

**2. The held-out rule.** Score on rows the model never trained on. Score on training rows and
you're measuring memory: with thousands of word-columns and hundreds of documents, a logistic
regression can nearly memorise the training set, and it will happily tell you 98 percent.

**3. The shape of the errors.** A confusion matrix says *which* side it gets wrong: rows are
the true labels, columns are the predictions, and the off-diagonal cells are the mistakes. A
model that's 60 percent accurate by calling almost everything pile A is a different animal from
one that's 60 percent accurate evenly. `classification_report` says the same thing with
precision and recall attached.

**Optional if you're quick:** `cross_val_score(clf, X, y, cv=StratifiedKFold(5, shuffle=True,
random_state=0))` refits on five different splits, and the spread tells you how much your single
accuracy was luck. Use `shuffle=True`: the comments arrive newest-first within each pile, so
unshuffled folds would be comparing different *weeks* rather than different samples — and the
score drops for a reason that has nothing to do with your model.

**Look at four numbers, in this order.**

- **The baseline.** What you get for guessing the bigger pile every time.
- **Your margin.** Accuracy minus baseline, in points. This is the result.
- **The confusion matrix.** Are the two error cells about equal, or is one much bigger?
- **The five-fold spread**, if you ran it. Your margin needs to be bigger than this.

**Then fill in whichever fits your numbers:**

> "Guessing the bigger pile scores ___%. Our model got ___% on rows it never saw. That is ___
> points better, and the five splits only varied by ___, so the two communities do write
> differently enough for a machine to tell."

> "Our model got ___% against a ___% baseline. That is only ___ points, about the same as the
> variation between splits. On this data we can't show the two communities write differently."

**Say this before anyone asks:** the margin says *that* they differ, not *how*. The how is
Station 5.

In [ ]:
# 4a. The floor. Fit DummyClassifier(strategy="most_frequent") on the same split and
#     score it, then print it next to your model's number.
# PROMPT IDEA: "fit a DummyClassifier most_frequent on Xtr, ytr and print its test score"


# BASELINE: ...    OUR MODEL: ...    DISTANCE ABOVE THE FLOOR: ...

In [ ]:
# 4b. Which errors? confusion_matrix(yte, predictions). Say out loud which cell is which
#     before you interpret it.


# WHICH SIDE IT GETS WRONG: ...

In [ ]:
# 4c. Optional: how lucky was that one split? Five shuffled folds, mean and spread.
# PROMPT IDEA: "cross_val_score with StratifiedKFold(5, shuffle=True, random_state=0),
#               print the mean and the standard deviation"

---

## Station 5 · Read its mind (5 min)

This is why today's model is a logistic regression and not something cleverer. Every word has
one signed weight; positive pushes toward one label, negative toward the other, and the largest
of each are the model's reasoning laid out in full. Nothing in Week 7's annotator will let you
do this.

**Get the two arrays and line them up.** `model.coef_.ravel()` is the weights,
`vectorizer.get_feature_names_out()` is the words, in the same column order.
`weights.argsort()` gives you the indices from most negative to most positive — take from both
ends. A horizontal bar chart of the top six each way (`plt.barh`) makes it a slide.

**Then do the part that's actually the lesson: sort the top words into three kinds.**

| Kind | Means | Example |
|---|---|---|
| **Topic** | the two piles talk about different things | *beaches*, *lebron*, *espresso* |
| **Register** | the two piles talk in different styles | *citation*, *lol*, *therefore* |
| **Community habit** | the two piles have different rituals | *this sub*, *mods*, *OP*, *edit* |

A model living on topic tells you the communities discuss different subjects — often obvious
before you started. A model living on register or habit is the more interesting claim: same
subject, different voice. **Which kind is yours, and does that answer the question you wrote at
Station 1?**

**Look at the words one at a time, not just the list.**

- **The ten each way, in order.** The order says which words mattered most.
- **How often each top word appears, and on which side.** A big weight can come from a rare
  word: four documents, all on one side, is enough to earn one.
- **Which kind each word is** — topic, register, or habit.
- **The words you predicted at Station 1 that aren't there.** What's missing counts too.

**Then fill this in:**

> "The words that separate ___ from ___ are mostly [topic / register / habit]: ___ against ___.
> So these communities [talk about different things / talk in different styles / do different
> things in their threads]. It does not show that ___."

That last line matters. If your top words are place names, you have shown people mention where
they live. You have not shown they write differently.

**Say this before anyone asks:** how many documents is your headline word in, and would it still
be there in next month's comments?

In [ ]:
# 5a. Line the words up against their weights and print the top ten each way.
# PROMPT IDEA: "using my fitted model and vectorizer, print the 10 words with the
#               largest positive and the 10 with the largest negative coefficients"


# TOP WORDS, SIDE A: ...
# TOP WORDS, SIDE B: ...

In [ ]:
# 5b. Make it a picture: a horizontal bar chart of the top six each way.

In [ ]:
# 5c. Spot-check one word you find surprising: how many documents is it actually in,
#     and on which side? (A large weight on four documents is a hunch, not a finding.)


# TOPIC / REGISTER / HABIT - our sort: ...
# THE ONE THAT SURPRISED US: ...

---

## Station 6 · Break it (4 min)

Your classifier has exactly two boxes and no concept of *neither*. Hand it a line of
Shakespeare, a recipe, a sentence in another language, and it will answer — confidently, with
one of your two labels and a probability. That property is not a bug in this notebook; it is
true of nearly every classifier deployed anywhere, and the confidence number does not warn you.

**Two things to do:**

1. **Out-of-domain.** Write a `predict(s)` helper around `model.predict_proba(vectorizer
   .transform([s]))[0, 1]` and feed it three inputs it has no business classifying. Note the
   probabilities. Anything near 0.5 means "no evidence either way" — the honest answer — but
   watch how often you get 0.8 on a sentence about nothing.
2. **A real error.** Find a document from the held-out half that it gets wrong, print it in
   full, and read it. Why did it fail? Is the document ambiguous even to you, is it short, or
   did one strong word drag it across?

**Look at the probabilities, not just the labels.**

- **What it says about text from neither pile.** Around 0.5 means "no evidence", which is the
  right answer. 0.8 on a sentence about nothing is the interesting case.
- **The probability on the real mistake.** A confident mistake and an unsure one are different
  problems, and only one of them could be caught by a threshold.
- **The document itself.** Is it short? Ambiguous to you as well? Pulled across by one word?

**Then fill this in:**

> "Given ___, which is from neither community, the model answered ___ with p = ___. It has no
> way to say 'neither'. To use it on unsorted text we would need ___ (a probability cutoff, a
> third class, or someone checking)."

That second move is Underwood's Pynchon misread, done on your own corpus. His genre classifier
called *The Crying of Lot 49* detective fiction, and the mistake is the most-cited thing about
the model, because it showed the boundary was fuzzy in a way the accuracy number never could.
**A classifier's errors are where it tells you what your categories actually are.**

In [ ]:
# 6a. Write a predict(s) helper: takes a string, prints the label and the probability.
# PROMPT IDEA: "write predict(s) that vectorizes one string and prints the predicted
#               label with predict_proba"

In [ ]:
# 6b. Feed it three inputs it has no business classifying. A line of Shakespeare, a
#     recipe step, a sentence in another language.


# WHAT IT SAID ABOUT THINGS IT COULDN'T KNOW: ...

In [ ]:
# 6c. Find a real held-out document it gets wrong and print it in full. Read it aloud.
# PROMPT IDEA: "show me the held-out rows where the prediction differs from the label,
#               with the full text"


# THE REAL ERROR, AND WHY WE THINK IT HAPPENED: ...

---

## Station 7 · The workbench: run the same question through many models (6 min)

One model is a result. **Several models are an argument.** You built the pipeline by hand at
Stations 2–5; from here the plumbing is written for you so the six minutes go on *choices*
rather than typing.

`fit_model(...)` fits one configuration, prints a compact report, and remembers it. `table()`
shows every fit side by side. The dials:

| Dial | Try | What it does |
|---|---|---|
| `vectorizer` | `"count"` / `"tfidf"` | Raw counts, or counts discounted by how many documents a word appears in. Week 2's tool. |
| `min_df` | `1`, `3`, `10` | A word must appear in at least this many documents to get a column. Drops one-off spellings, names and typos the model could memorise. |
| `ngram` | `(1,1)`, `(1,2)` | Add two-word features, so *this sub* is one column instead of two. Multiplies the vocabulary — pair it with `min_df`. |
| `C` | `0.05`, `1`, `20` | How hard the model is pushed toward small weights. Low = simpler and usually generalises better; high = it trusts the training rows and overfits faster. |
| `class_weight` | `None`, `"balanced"` | Make errors on the smaller pile cost more, so the model stops riding whichever side is bigger. |
| `stop_words` | `None`, `"english"` | Throw away the function words. Watch what happens to the score AND the word lists — they often disagree. |

**Change exactly one thing at a time.** Four fits minimum, and `table()` at the end.

Two questions the table has to answer, and they are what you'll be asked at report-back:

1. **Does the held-out accuracy actually move?** If four quite different models land within a
   point or two of each other, your modelling choices weren't what decided the answer. The
   corpus was — and that is a finding, not a failure.
2. **Do the top words change more than the accuracy does?** This is the common and unsettling
   result: same score, visibly different explanation. It means the score was never the finding,
   and anyone reporting only accuracy would have hidden the disagreement.

Watch the `train` column too. When it sits at 1.00 while `heldout` sits near 0.60, the model
has memorised the training rows — that gap *is* overfitting, in two numbers.

**Read the table down the columns, not across the rows.**

- **`heldout`: best minus worst.** Compare that range to the five-fold spread from Station 4.
  If it's smaller, your settings didn't decide anything.
- **`features`.** How much vocabulary each setting threw away. `min_df=10` often cuts most of
  the columns and barely moves the score.
- **`train` minus `heldout`, row by row.** Which setting controls the memorising.
- **`top_A` and `top_B` down the column.** Do the same words survive every setting?

**Then fill in whichever fits:**

> "Across ___ settings the accuracy went from ___ to ___, a range of ___ points. That is
> [smaller / bigger] than the variation between splits, so the settings [didn't / did] decide
> the answer."

> "The accuracy barely moved, but the top words did: ___ leads with plain counts and disappears
> with tf-idf. Reporting only the accuracy would have hidden that."

**Say this before anyone asks:** you picked which settings to try. Say which ones you swept, not
just which one won.

In [ ]:
# --- The workbench. Provided, because the choices are the lesson, not the typing. ---
RESULTS, LAST = [], {}

def _build(vectorizer, min_df, ngram, stop_words, max_features, sublinear_tf, binary):
    kw = dict(min_df=min_df, ngram_range=ngram, stop_words=stop_words, max_features=max_features)
    if vectorizer.startswith("tf"):
        return TfidfVectorizer(sublinear_tf=sublinear_tf, **kw)
    return CountVectorizer(binary=binary, **kw)

def fit_model(name=None, vectorizer="count", min_df=1, ngram=(1, 1), C=1.0,
              class_weight=None, stop_words=None, max_features=None,
              sublinear_tf=True, binary=False, show=True):
    """Fit ONE configuration on the current corpus, report it, and remember it."""
    vec = _build(vectorizer, min_df, ngram, stop_words, max_features, sublinear_tf, binary)
    X = vec.fit_transform(df["text"])
    y = (df["label"] == LABEL_A).astype(int).values
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
    clf = LogisticRegression(C=C, class_weight=class_weight, max_iter=3000).fit(Xtr, ytr)
    acc, train_acc = clf.score(Xte, yte), clf.score(Xtr, ytr)
    base = DummyClassifier(strategy="most_frequent").fit(Xtr, ytr).score(Xte, yte)
    full = LogisticRegression(C=C, class_weight=class_weight, max_iter=3000).fit(X, y)
    words, w = np.array(vec.get_feature_names_out()), full.coef_.ravel()
    order = w.argsort()
    top_a, top_b = list(words[order[-6:][::-1]]), list(words[order[:6]])
    if name is None:
        name = (f"{vectorizer} min_df={min_df} ngram={ngram[0]}-{ngram[1]} C={C}"
                + (" balanced" if class_weight else "")
                + (" no-stopwords" if stop_words else ""))
    RESULTS.append(dict(name=name, corpus=f"{LABEL_A}/{LABEL_B}", features=X.shape[1],
                        baseline=round(base, 3), train=round(train_acc, 3),
                        heldout=round(acc, 3), top_A=", ".join(top_a[:3]),
                        top_B=", ".join(top_b[:3])))
    LAST.update(dict(name=name, vec=vec, clf=full, words=words, w=w))
    if show:
        print(name)
        print(f"  {X.shape[1]:>6,} features   baseline {base:.2f}   held-out {acc:.2f}"
              f"   (on its own training rows {train_acc:.2f})")
        print(f"  most {LABEL_A}: {', '.join(top_a)}")
        print(f"  most {LABEL_B}: {', '.join(top_b)}")
    return RESULTS[-1]

def table():
    """Every fit so far, side by side. This table IS your argument."""
    if not RESULTS:
        print("Nothing fitted yet - run fit_model(...) first.")
        return None
    return pd.DataFrame(RESULTS)

# Your first fit, so the table has something in it. Then change ONE thing and refit.
fit_model()

In [ ]:
# YOUR SWEEP. One change per line - keep the rest fixed, or you cannot say what moved.
# fit_model(min_df=3)
# fit_model(vectorizer="tfidf", min_df=3)
# fit_model(min_df=3, ngram=(1, 2))
# fit_model(min_df=3, C=0.05)
# fit_model(min_df=3, C=20)
# fit_model(min_df=3, class_weight="balanced")
# fit_model(min_df=3, stop_words="english")

table()

# DOES THE ACCURACY MOVE: ...
# DO THE TOP WORDS MOVE MORE THAN THE ACCURACY: ...
# THE MODEL WE'D KEEP, AND WHY (not "it scored highest"): ...

In [ ]:
# Or write the sweep yourself, without fit_model(), if you would rather see every step.
# A loop over a list of settings, fitting and printing each: it is ten lines, and the
# group that writes it understands the table better than the group that calls it.
# PROMPT IDEA: "loop over min_df in [1, 3, 10], fit a logistic regression for each,
#               and print the held-out accuracy and top 5 words per class"

### The same sweep with sliders, if you prefer

`workbench()` draws the dials as controls: set them, press **Fit**, and the row lands in the
same `table()`. Nothing here you can't do by typing `fit_model(...)` — some groups think faster
with a slider in hand, and the point is to run *many* configurations in six minutes.

In [ ]:
try:
    import ipywidgets as W
    from IPython.display import display, clear_output
    HAVE_WIDGETS = True
except Exception:
    HAVE_WIDGETS = False

def workbench():
    """Sliders and dropdowns over fit_model(). Falls back to a printed hint."""
    if not HAVE_WIDGETS:
        print("ipywidgets isn't available here - use fit_model(min_df=3, C=0.05, ...) directly.")
        return
    v = W.Dropdown(options=["count", "tfidf"], value="count", description="features")
    mdf = W.IntSlider(value=1, min=1, max=15, description="min_df")
    ng = W.Dropdown(options=[("single words", (1, 1)), ("words + pairs", (1, 2))],
                    value=(1, 1), description="n-grams")
    c = W.SelectionSlider(options=[0.01, 0.05, 0.2, 1, 5, 20, 100], value=1, description="C")
    cw = W.Dropdown(options=[("as they come", None), ("balanced", "balanced")],
                    value=None, description="classes")
    sw = W.Dropdown(options=[("keep them", None), ("drop english stopwords", "english")],
                    value=None, description="stopwords")
    go, out = W.Button(description="Fit and add to the table", button_style=""), W.Output()
    def _fit(_):
        with out:
            clear_output()
            fit_model(vectorizer=v.value, min_df=mdf.value, ngram=ng.value, C=c.value,
                      class_weight=cw.value, stop_words=sw.value)
            print(f"\n{len(RESULTS)} fits so far - call table() to see them all.")
    go.on_click(_fit)
    display(W.VBox([W.HBox([v, mdf, ng]), W.HBox([c, cw, sw]), go, out]))

workbench()

---

## Station 8 · Interesting weights: is that word a finding, or a giveaway? (5 min)

Look at your top words. On the default pair, the list will be led by things like *diego*, *sd*,
*bay*, *oakland* — place names. The model found the giveaway, which is correct and completely
uninteresting: of course a comment mentioning Oakland is from the Bay Area room.

**The interesting question starts one step later: what does it find when the giveaway is gone?**

`why("word")` tells you, for the model you fitted last: the weight, its rank among all features,
how many documents it appears in split by pile, and a couple of those documents so you can read
the word in use. Use it on three words — the top one, one that surprises you, and one you
suspect is junk (a number, a typo, half a URL).

Then knock the obvious ones out and refit:

```python
GIVEAWAYS = ["diego", "sd", "bay", "oakland", "sf", "francisco", "san", "jose", "berkeley"]
fit_model(stop_words=GIVEAWAYS, min_df=3)
```

The accuracy will drop. **The question is how far.** If it collapses to the baseline, the two
communities differed only in the names they say, and you have learned that. If it holds most of
its ground, something else is separating them — and *that* list of words, the one underneath the
place names, is the finding worth reporting.

**Look at three things.**

- **Accuracy before and after**, both against the same baseline.
- **How much of your margin is left.** 0.77 down to 0.70 with a 0.50 baseline means you kept
  about two thirds. Down to 0.52 means you kept nothing.
- **The new top words.** This is the list you would actually put in a write-up.

**Then fill in whichever fits:**

> "With the place names in, the model gets ___%. Take them out and it still gets ___% against a
> ___% baseline. So apart from naming their own cities, the two communities differ in ___."

> "Take the place names out and the model drops to the baseline. What separated these two
> communities was that they say their own place names. Otherwise they write alike."

`contributions("...")` runs the same idea forward: type any sentence and see each word's
count × weight, which words the model ignored entirely, and how the probability was assembled.
Try a sentence about your two communities, then something from neither — a recipe, a line of
Shakespeare — and watch it answer confidently anyway.

In [ ]:
# --- Provided: two ways to look inside the model you last fitted ---------------
def _docs_with(word):
    return df[df["text"].str.contains(rf"(?<!\w){re.escape(word)}(?!\w)", case=False, regex=True)]

def why(word, examples=2, chars=150):
    """Weight, rank, which pile, and the word in use."""
    if not LAST:
        print("Fit a model first (Station 7).")
        return
    words, w = LAST["words"], LAST["w"]
    idx = np.where(words == word.lower())[0]
    if not len(idx):
        print(f"'{word}' is not a column in this model: dropped by min_df or stop_words, "
              f"or it never appears in this corpus.")
        return
    i = int(idx[0]); weight = w[i]
    rank = int((np.abs(w) > abs(weight)).sum()) + 1
    hit = _docs_with(word)
    print(f"'{word}'  weight {weight:+.3f}  ->  pushes toward {LABEL_A if weight > 0 else LABEL_B}")
    print(f"  rank {rank} of {len(w):,} by size of weight")
    print(f"  in {len(hit)} of {len(df)} documents: "
          + ", ".join(f"{v} {k}" for k, v in hit["label"].value_counts().items()))
    if len(hit) < 8:
        print("  ^^ that is a handful of documents. A big weight on this little evidence is "
              "a hunch, not a finding.")
    for lab in (LABEL_A, LABEL_B):
        for t in hit[hit["label"] == lab]["text"].head(examples):
            s_ = t.replace("\n", " ")
            m = re.search(rf"(?<!\w){re.escape(word)}(?!\w)", s_, re.I)
            if m:
                a = max(0, m.start() - chars // 2)
                s_ = ("..." if a else "") + s_[a:a + chars] + "..."
            print(f"    [{lab}] {s_}")

def contributions(sentence, k=8):
    """How the decision was assembled: each word's count x its weight."""
    if not LAST:
        print("Fit a model first (Station 7).")
        return
    vec, clf, words = LAST["vec"], LAST["clf"], LAST["words"]
    v = vec.transform([sentence]); w = clf.coef_.ravel()
    contrib = v.multiply(w).toarray().ravel()
    nz = [i for i in np.argsort(-np.abs(contrib))[:k] if contrib[i] != 0]
    p = clf.predict_proba(v)[0, 1]
    print(f"{sentence!r}")
    print(f"  -> {LABEL_A if p > 0.5 else LABEL_B}   p_{LABEL_A} = {p:.2f}   "
          f"(these weights, plus an intercept of {clf.intercept_[0]:+.2f})")
    if not nz:
        print("  not one word of this is in the vocabulary - it is answering from the "
              "intercept alone, and it still answers.")
    for i in nz:
        bar = "#" * max(1, int(abs(contrib[i]) / abs(contrib[nz]).max() * 20))
        toward = f"-> {LABEL_A}" if contrib[i] > 0 else f"<- {LABEL_B}"
        print(f"  {words[i]:>20}  {contrib[i]:+.3f}  {toward:>14}  {bar}")
    unseen = [t for t in re.findall(r"\w+", sentence.lower()) if t not in set(words)]
    if unseen:
        print(f"  ignored entirely (not in the vocabulary): {', '.join(unseen[:10])}")

# The model's own top word, to start you off:
why(RESULTS[-1]["top_A"].split(", ")[0])

In [ ]:
# YOUR THREE WORDS. The top one, a surprising one, one you suspect is junk.
# why("...")
# why("...")
# why("...")

# NOW KNOCK OUT THE GIVEAWAYS AND REFIT. List the words that were never in doubt:
# GIVEAWAYS = ["diego", "sd", "bay", "oakland", "sf", "francisco", "san"]
# fit_model(stop_words=GIVEAWAYS, min_df=3, name="place names removed")
# table()

# And a sentence or two through contributions():
# contributions("something a person in one of your two communities would write")
# contributions("shall i compare thee to a summer's day")

# ACCURACY BEFORE / AFTER REMOVING THE GIVEAWAYS: ...
# WHAT THE MODEL FOUND UNDERNEATH THEM: ...

In [ ]:
# Your own follow-up. Peel a second layer, count how often a suspect word appears per
# pile by hand, check whether one thread produced it - whatever the last cell made you
# want to know. This one is entirely yours.

### The same two tools as boxes to type in

In [ ]:
def word_box():
    """Type a word, see everything the corpus knows about it."""
    if not HAVE_WIDGETS:
        print("Use why(\"word\") directly.")
        return
    t = W.Text(value="", placeholder="a word from the weight list", description="word")
    go, out = W.Button(description="Interrogate it"), W.Output()
    def _go(_):
        with out:
            clear_output(); why(t.value.strip())
    go.on_click(_go)
    display(W.VBox([W.HBox([t, go]), out]))

def sentence_box():
    """Type a sentence, watch the model assemble its answer."""
    if not HAVE_WIDGETS:
        print("Use contributions(\"your sentence\") directly.")
        return
    t = W.Text(value="", placeholder="any sentence at all", description="sentence",
               layout=W.Layout(width="60%"))
    go, out = W.Button(description="Break it down"), W.Output()
    def _go(_):
        with out:
            clear_output(); contributions(t.value.strip())
    go.on_click(_go)
    display(W.VBox([W.HBox([t, go]), out]))

word_box()
sentence_box()

---

## Station 9 · Change the corpus, not the model (5 min)

Everything so far varied the *model* and held the *data* fixed. Now do the opposite, which is
the harder test and the one nobody runs: keep your best configuration exactly as it is, and
point it at two different communities.

Pick a pair you actually know. Some that teach well:

| Pair | The question it asks |
|---|---|
| `("coffee", "espresso")` | a hobby and its more specialised neighbour — is the difference topic, or expertise? |
| `("AskMen", "AskWomen")` | same format, different rooms |
| `("nba", "soccer")` | two sports: an easy pair, and a useful contrast with your hard one |
| `("Seattle", "Portland")` | two cities, the same shape of problem as the default |
| `("cats", "dogs")` | you will get a high score. Is that a finding? |

`use_pair("a", "b")` reloads the corpus and keeps your results table, tagging the new rows with
the new corpus so you can read them side by side. Then refit **the same configuration** and
compare.

**Look at these, with the settings left alone.**

- **Accuracy and baseline.** Both change on a new pair, so compare the *margin*.
- **What kind of word wins.** Topic on one pair and register on another tells you about the
  pairs, not the model.
- **Whether your best setting is still the best one.** Often it isn't.

**Then fill in whichever fits:**

> "The same settings beat the baseline by ___ points on ___ and ___ points on ___, and both
> times the words are mostly [kind]. The method carries over. The words don't."

> "On ___ the same model barely beats the baseline. What we found on ___ was about those two
> communities, not about communities in general."

This is the cheapest check in the course, and most published work skips it.

In [ ]:
def use_pair(a, b, n=N_PER_SIDE):
    """Swap in two different communities, keeping the results table."""
    global df, LABEL_A, LABEL_B
    df, LABEL_A, LABEL_B = load_pair((a, b), n=n)
    print(f"\ncorpus is now {LABEL_A} vs {LABEL_B}: {len(df)} rows. "
          f"Refit with the SAME settings and compare.")
    return df

def pair_box():
    if not HAVE_WIDGETS:
        print('Use use_pair("coffee", "espresso") directly.')
        return
    a = W.Text(value="coffee", description="pile A")
    b = W.Text(value="espresso", description="pile B")
    go, out = W.Button(description="Load this pair"), W.Output()
    def _go(_):
        with out:
            clear_output(); use_pair(a.value.strip(), b.value.strip())
    go.on_click(_go)
    display(W.VBox([W.HBox([a, b, go]), out]))

pair_box()

# Or in one line, then refit the configuration you kept:
# use_pair("coffee", "espresso")
# fit_model(min_df=3)          # <- the SAME settings as your best run above
# table()

# DID THE ACCURACY SURVIVE THE NEW PAIR: ...
# DID THE KIND OF WORD (topic / register / habit) STAY THE SAME: ...

In [ ]:
# Your pair, your configuration, your comparison. Load it, refit it, and print the two
# results next to each other so the difference is visible rather than remembered.

---

## Open bench · The question your group actually wants to ask

Three empty cells and no instructions. This is where a project usually starts.

Things groups have done here: added a third community and looked at which pair the model
confuses; pulled the same subreddit from two different years and compared the weights; hand-
labelled a hundred rows and classified something nobody had tagged; checked whether the model's
confidence means anything by scoring only the documents it was surest about.

Whatever you run: predict first, then run it, then ask what it actually showed. If a cell does
something you didn't expect, that's the useful part.

---

## Station 10 · Report back (4 min)

Ninety seconds a group. The ten answers together are the session's actual finding, so write
them down rather than improvising.

- **Our two piles:** \_\_\_ vs \_\_\_ , chosen because \_\_\_
- **Baseline / our best model:** \_\_\_ % / \_\_\_ %
- **Configurations we ran:** \_\_\_ (and the one that mattered: \_\_\_)
- **Did the accuracy move, or only the words?** \_\_\_
- **Top words, side A:** \_\_\_ **side B:** \_\_\_
- **Topic, register, or habit?** \_\_\_
- **What survived when we removed the giveaway words:** \_\_\_
- **Where it broke:** \_\_\_
- **What happened on a second pair of communities:** \_\_\_
- **One caveat we would put in writing:** \_\_\_

The last three are what the room will argue about. A caveat is not a disclaimer: it is the
specific thing that would have to be checked before your sentence is safe to publish — *"our
headline word appears in eleven documents; we didn't check whether one thread produced them."*

**Before you stand up**, check your sentence has the four parts from the front of the notebook:
the claim, the number, what it covers, and the objection you raise yourself. Ninety seconds is
enough for that and nothing else.

In [ ]:
# Save the session so it survives the runtime (and so the sketch homework is easy).
table().to_csv(os.path.join(PROJECT_DIR, "week03_results.csv"), index=False)
print("saved to", os.path.join(PROJECT_DIR, "week03_results.csv"))

# Optional: the top weights of your kept model, as a chart for the report-back.
# w, words = LAST["w"], LAST["words"]
# top = np.concatenate([w.argsort()[:6], w.argsort()[-6:]])
# plt.figure(figsize=(6, 3)); plt.barh(words[top], w[top])
# plt.title(f"{LAST['name']}"); plt.tight_layout(); plt.show()

---

### If you finish early, or want to keep going after class

- **A third pile.** Add a community and refit; logistic regression handles three classes (one
  set of weights per class), and the confusion matrix starts telling you which pair it *can't*
  separate.
- **How much data do you actually need?** Refit on 50, 100, 200, 400 rows a side and plot
  accuracy against corpus size. Most curves flatten sooner than people expect — useful to know
  before you spend Week 4 collecting.
- **Peel the giveaways twice.** Remove the place names, refit, then remove the *new* top words
  and refit again. How many layers deep does the difference between two communities go before
  the model is down to the baseline? That is a real experiment, and nobody has run it on your
  pair.
- **The same pair, a year apart.** Pull each side from two different time windows and compare
  the weights. Communities drift: a Week 2 trend question with a Week 3 tool.
- **Your own labels.** The sketch homework: point `fit_model` at any labelled set you care
  about — your own tagged photos, emails, reviews — by building a `df` with `text` and `label`
  columns. Everything in this notebook works unchanged.

---

### Cheat sheet

| You want | Reach for |
|---|---|
| text → matrix | `CountVectorizer()`, `TfidfVectorizer()`, `.fit_transform(texts)` |
| the column names | `vectorizer.get_feature_names_out()` |
| hold data out | `train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)` |
| fit | `LogisticRegression(max_iter=1000).fit(Xtr, ytr)` |
| score | `model.score(Xte, yte)` or `accuracy_score(yte, model.predict(Xte))` |
| the floor | `DummyClassifier(strategy="most_frequent")` |
| which errors | `confusion_matrix(yte, preds)`, `classification_report(yte, preds)` |
| how lucky was that | `cross_val_score(model, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=0))` |
| the weights | `model.coef_.ravel()`, then `.argsort()` |
| a probability | `model.predict_proba(vectorizer.transform([s]))[0, 1]` |
| **many configurations** | `fit_model(min_df=3, C=0.05, ...)`, then `table()` |
| **one word, interrogated** | `why("word")` · `word_box()` |
| **how a decision was assembled** | `contributions("a sentence")` · `sentence_box()` |
| **a different pair of communities** | `use_pair("coffee", "espresso")` · `pair_box()` |

**Stuck?** Ask the AI for the piece, not the notebook. Predict, run, interrogate — every cell,
all term. The worked version of this pipeline is `week03_classification.ipynb`; errors are in
`../kits/common-errors-cheatsheet.md`.

### The homework this feeds

Your sketch is one of today's models on a labelled set *you* are curious about, plus a
screenshot of its five most positive and five most negative words — do they make sense? And
bring your **corpus existence proof** to Week 4: a screenshot of 50 loadable rows of the data
you want to use. No proof, no pitch.